# 03 — Streaming Logic Preview


**Course:** Big Data Management Systems and Tools  
**Project:** Batch vs. Streaming Analytics at Scale (Apache Spark)  
**Dataset:** City of Chicago — Traffic Crashes – Crashes (Socrata ID: 85ca-t3if)  

---

## Purpose

This notebook **explains and validates** the streaming pipeline. It is **not** an orchestration notebook.

**What this notebook does:**
- Documents the event-time window definition and output schema used by `streaming_job.py`.
- Reads already-produced streaming outputs **as batch** (no streaming API, no `awaitTermination`) to validate correctness.
- Verifies hourly window alignment, schema structure, and sanity invariants.
- Provides the evidence checklist for the batch vs. streaming comparison report.

**What this notebook does NOT do:**
- Run `dropper.py` or `streaming_job.py`.
- Produce new aggregations or ETL logic.
- Assert result equality with batch (streaming covers only replayed chunks; batch has full history).

---

## Prerequisites - create replay files

Before running this notebook, you must have **already executed the streaming pipeline** to produce outputs under `outputs/streaming/`. This involves:
1. Running `src/make_replay_files.py` once to create day-chunked Parquet files under `data/replay/`.

-To run it, execute the following command in the terminal: 
```
python src\make_replay_files.py `
  --input data\silver\crashes_parquet `
  --output data\replay `
  --chunking day `
  --overwrite
```

--- 
## How to Produce Streaming Outputs

Run **both scripts from separate terminals** before executing this notebook:

```
# Terminal 1 — start the streaming consumer
python src/streaming_job.py `
  --input   data/stream_in `
  --output  outputs/streaming `
  --checkpoint data/checkpoints/streaming_job `
  --silver  data/silver/crashes_parquet

# Terminal 2 — start the file dropper (simulates real-time arrival)
python src/dropper.py `
  --source data/replay `
  --dest   data/stream_in `
  --interval-seconds 5
```

Then this notebook (to inspect and validate the outputs).

---

## Streaming Pipeline Overview

```
+------------------------+         +-------------------+         +----------------------+
|  data/replay/          |         |  data/stream_in/  |         |  outputs/streaming/  |
|  (Parquet chunks,      | drop()  |  (watched by      |  Spark  |  (hourly window      |
|  partitioned by        +-------> |  streaming_job)   +-------> |  aggregates,         |
|  chunk_key=YYYY-MM-DD) |         |                   |  Str.   |  Parquet)            |
+------------------------+         +-------------------+         +----------------------+
         ^                                  ^                              ^
make_replay_files.py                  dropper.py                 streaming_job.py
```

| Script | Role |
|---|---|
| `src/make_replay_files.py` | One-time offline job — splits Silver into day-chunks under `data/replay/` |
| `src/dropper.py` | Simulates real-time file arrival — copies one chunk every N seconds into `data/stream_in/` |
| `src/streaming_job.py` | Spark Structured Streaming job — consumes `data/stream_in/`, emits hourly windows to `outputs/streaming/` |

---

## Event-Time Window Definition

| Parameter | Value |
|---|---|
| Event-time column | `CRASH_DATE` (timestamp) |
| Window type | Fixed, non-overlapping (tumbling) |
| Window duration | **1 hour** |
| Watermark | 1 hour (late-data tolerance) |
| Output mode | `append` |

This is the **same window definition** used in `02_batch_analysis.ipynb` (`crashes_by_hour_window`), making the outputs directly comparable.

## Output Schema: `crashes_by_hour_window`

| Column | Type | Description |
|---|---|---|
| `window_start` | timestamp | Inclusive start of the 1-hour bucket |
| `window_end` | timestamp | Exclusive end of the 1-hour bucket |
| `crashes` | long | Count of crash records in the window |
| `injuries_total` | long | Sum of total injuries |
| `injuries_fatal` | long | Sum of fatal injuries |

In [1]:
import os
import glob
import findspark

# ── Windows: set Hadoop/Java env vars before PySpark imports ─────────────────
os.environ.setdefault("JAVA_HOME",   r"C:\Program Files\Eclipse Adoptium\jdk-11.0.30.7-hotspot")
os.environ.setdefault("HADOOP_HOME", r"C:\hadoop")
os.environ["PATH"] = os.environ["HADOOP_HOME"] + r"\bin" + os.pathsep + os.environ.get("PATH", "")

findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ── SparkSession (no explicit memory — lightweight validation only) ────────────
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("streaming_preview")
    .config("spark.sql.repl.eagerEval.enabled", True)
    .getOrCreate()
)

# ── Path constants ─────────────────────────────────────────────────────────────
NOTEBOOK_DIR      = os.path.dirname(os.path.abspath("__file__"))
PROJECT_ROOT      = os.path.join(NOTEBOOK_DIR, "..")
STREAMING_OUT     = os.path.join(PROJECT_ROOT, "outputs", "streaming")
BATCH_WINDOW_OUT  = os.path.join(PROJECT_ROOT, "outputs", "batch", "crashes_by_hour_window")
SILVER_PATH       = os.path.join(PROJECT_ROOT, "data", "silver", "crashes_parquet")
STREAM_IN         = os.path.join(PROJECT_ROOT, "data", "stream_in")
CHECKPOINT_PATH   = os.path.join(PROJECT_ROOT, "data", "checkpoints", "streaming_job")

print(f"Spark {spark.version}  ready.")

Spark 3.4.0  ready.


In [2]:
# ── Configuration snapshot — audit record for the report ──────────────────────
WINDOW_SIZE = "1 hour"
WATERMARK   = "1 hour"
TRIGGER     = "10 seconds"   # informational; streaming_job.py uses this, not this notebook

print("=" * 60)
print("  STREAMING JOB — CONFIGURATION SNAPSHOT")
print("=" * 60)
print(f"  Window size          : {WINDOW_SIZE}")
print(f"  Watermark            : {WATERMARK}")
print(f"  Trigger interval     : {TRIGGER}")
print(f"  Event-time column    : CRASH_DATE")
print(f"  Output mode          : append")
print()
print(f"  Silver path          : {os.path.abspath(SILVER_PATH)}")
print(f"  Streaming input      : {os.path.abspath(STREAM_IN)}")
print(f"  Streaming output     : {os.path.abspath(STREAMING_OUT)}")
print(f"  Batch window output  : {os.path.abspath(BATCH_WINDOW_OUT)}")
print(f"  Checkpoint path      : {os.path.abspath(CHECKPOINT_PATH)}")
print("=" * 60)

  STREAMING JOB — CONFIGURATION SNAPSHOT
  Window size          : 1 hour
  Watermark            : 1 hour
  Trigger interval     : 10 seconds
  Event-time column    : CRASH_DATE
  Output mode          : append

  Silver path          : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\silver\crashes_parquet
  Streaming input      : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\stream_in
  Streaming output     : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\outputs\streaming
  Batch window output  : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\outputs\batch\crashes_by_hour_window
  Checkpoint path      : c:\Users\miguel.morales\GIT\AI Certificate\Big Data\big-data_group-10\data\checkpoints\streaming_job


## Step 1 — Load Streaming Output (Batch Read)

The streaming job writes Parquet files to `outputs/streaming/` after each micro-batch. This cell reads all files produced so far **as a static batch** — no streaming API is used.

**If no output is found yet:** the cell prints a diagnostic message showing the current state of the pipeline (files queued in `data/stream_in/`, checkpoint presence) so you know exactly what to do next.

> **Note:** This section is safe to re-run at any time after the streaming job has produced data.

In [3]:
# Recursive glob — detect actual Parquet data files (not just folder existence)
streaming_parquet_files = glob.glob(
    os.path.join(STREAMING_OUT, "**", "*.parquet"), recursive=True
)

if not streaming_parquet_files:
    # ── Diagnostic guard ──────────────────────────────────────────────────────
    print("⚠  No streaming output found in outputs/streaming/")
    print()

    # How many files are queued in stream_in?
    stream_in_files = glob.glob(os.path.join(STREAM_IN, "**", "*.parquet"), recursive=True)
    print(f"  Files queued in data/stream_in/ : {len(stream_in_files)}")

    # Is checkpoint present?
    ckpt_exists = os.path.isdir(CHECKPOINT_PATH) and bool(os.listdir(CHECKPOINT_PATH))
    print(f"  Checkpoint exists               : {'Yes' if ckpt_exists else 'No'}")

    print()
    print("  ➜  Start streaming_job.py and dropper.py (see notebook header),")
    print("     then re-run this cell once output files appear.")
    stream_df = None
else:
    # ── Load and inspect ──────────────────────────────────────────────────────
    stream_df = spark.read.parquet(*streaming_parquet_files)
    row_count = stream_df.count()

    print(f"  Parquet files found : {len(streaming_parquet_files)}")
    print(f"  Rows loaded         : {row_count:,}")
    print()
    print("Schema:")
    stream_df.printSchema()
    print("Sample rows (ordered by window_start):")
    stream_df.orderBy("window_start").show(10, truncate=False)

  Parquet files found : 465
  Rows loaded         : 745

Schema:
root
 |-- crashes: long (nullable = true)
 |-- injuries_total: long (nullable = true)
 |-- injuries_fatal: long (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)

Sample rows (ordered by window_start):
+-------+--------------+--------------+-------------------+-------------------+
|crashes|injuries_total|injuries_fatal|window_start       |window_end         |
+-------+--------------+--------------+-------------------+-------------------+
|12     |5             |0             |2022-01-01 00:00:00|2022-01-01 01:00:00|
|19     |3             |0             |2022-01-01 01:00:00|2022-01-01 02:00:00|
|21     |11            |0             |2022-01-01 02:00:00|2022-01-01 03:00:00|
|19     |11            |0             |2022-01-01 03:00:00|2022-01-01 04:00:00|
|13     |3             |0             |2022-01-01 04:00:00|2022-01-01 05:00:00|
|9      |0             |0        

## Step 2 — Window Semantics Validation

Each output row represents a **fixed, non-overlapping 1-hour bucket** anchored to wall-clock hours (e.g., `2022-01-01 14:00:00` → `2022-01-01 15:00:00`).

Two invariants must hold for every row:
1. `window_start` must be aligned to the top of an hour (`minute == 0`, `second == 0`).
2. `window_end − window_start` must equal exactly **3 600 seconds** (1 hour).

These are the same semantics as the batch `crashes_by_hour_window` output, enabling direct comparison.

In [4]:
if stream_df is None:
    print("⚠  No streaming output loaded — run Step 1 first.")
else:
    results = []

    # ── Check 1: window_start aligned to top-of-hour ──────────────────────────
    misaligned = stream_df.filter(
        (F.minute("window_start") != 0) | (F.second("window_start") != 0)
    )
    bad_align = misaligned.count()
    results.append((
        "window_start aligned to top-of-hour",
        bad_align == 0,
        f"misaligned={bad_align}"
    ))
    if bad_align > 0:
        print("  Examples of misaligned windows:")
        misaligned.select("window_start", "window_end", "crashes").show(5, truncate=False)

    # ── Check 2: window duration == exactly 1 hour (3 600 s) ─────────────────
    wrong_dur = stream_df.filter(
        (F.unix_timestamp("window_end") - F.unix_timestamp("window_start")) != 3600
    )
    bad_dur = wrong_dur.count()
    results.append((
        "window duration is exactly 1 hour",
        bad_dur == 0,
        f"bad_windows={bad_dur}"
    ))

    # ── Print results ─────────────────────────────────────────────────────────
    print("Window Semantics Validation")
    print("-" * 55)
    for label, passed, detail in results:
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"  {status}  {label}  ({detail})")

    # ── Time range covered ────────────────────────────────────────────────────
    min_ws, max_ws = stream_df.agg(F.min("window_start"), F.max("window_start")).first()
    print()
    print(f"  Earliest window : {min_ws}")
    print(f"  Latest window   : {max_ws}")

Window Semantics Validation
-------------------------------------------------------
  ✓ PASS  window_start aligned to top-of-hour  (misaligned=0)
  ✓ PASS  window duration is exactly 1 hour  (bad_windows=0)

  Earliest window : 2022-01-01 00:00:00
  Latest window   : 2022-02-01 21:00:00


## Step 3 — Sanity Checks

Basic data-quality invariants that must hold for the streaming output to be considered correct:

| Check | Expected |
|---|---|
| Row count | > 0 (output is non-empty) |
| `window_start` nulls | 0 (every row belongs to a window) |
| `injuries_total` / `injuries_fatal` negatives | 0 (physical impossibility) |
| `crashes` per window | ≥ 1 (empty windows are not written in `append` mode) |

In [5]:
if stream_df is None:
    print("⚠  No streaming output loaded — run Step 1 first.")
else:
    total_rows    = stream_df.count()
    null_starts   = stream_df.filter(F.col("window_start").isNull()).count()
    neg_injuries  = stream_df.filter(
        (F.col("injuries_total") < 0) | (F.col("injuries_fatal") < 0)
    ).count()
    zero_crashes  = stream_df.filter(F.col("crashes") < 1).count()

    checks = [
        ("Row count > 0",                   total_rows > 0,    f"rows={total_rows:,}"),
        ("No null window_start",             null_starts == 0,  f"nulls={null_starts}"),
        ("No negative injury values",        neg_injuries == 0, f"bad={neg_injuries}"),
        ("All windows have crashes >= 1",    zero_crashes == 0, f"bad={zero_crashes}"),
    ]

    all_passed = all(p for _, p, _ in checks)
    print("Sanity Checks")
    print("-" * 55)
    for label, passed, detail in checks:
        status = "✓ PASS" if passed else "✗ FAIL"
        print(f"  {status}  {label}  ({detail})")
    print()
    print("Overall:", "✓ ALL CHECKS PASSED" if all_passed else "✗ ONE OR MORE CHECKS FAILED")

Sanity Checks
-------------------------------------------------------
  ✓ PASS  Row count > 0  (rows=745)
  ✓ PASS  No null window_start  (nulls=0)
  ✓ PASS  No negative injury values  (bad=0)
  ✓ PASS  All windows have crashes >= 1  (bad=0)

Overall: ✓ ALL CHECKS PASSED


## Step 4 — Structural & Semantic Comparison with Batch Output

This section compares the streaming output against the corresponding **batch Gold table** (`outputs/batch/crashes_by_hour_window`).

**Scope: structural and semantic only — not result equality.**

| Dimension | What we check |
|---|---|
| Schema | Same column names and types in both outputs |
| Window alignment | Both use the same hourly tumbling window on `CRASH_DATE` |
| Row counts | Streaming covers only replayed chunks; batch has full 2022–2025 history |

Row counts will differ intentionally: the streaming job is a partial replay, not a full re-computation. Once all chunks have been dropped, totals should converge.

In [6]:
# ── Batch side ────────────────────────────────────────────────────────────────
batch_parquet_files = glob.glob(
    os.path.join(BATCH_WINDOW_OUT, "**", "*.parquet"), recursive=True
)

if not batch_parquet_files:
    print("⚠  Batch output not found. Run 02_batch_analysis.ipynb first.")
    batch_df = None
else:
    batch_df = spark.read.parquet(BATCH_WINDOW_OUT)
    batch_count = batch_df.count()
    print("=== BATCH — crashes_by_hour_window ===")
    print(f"  Rows (full history) : {batch_count:,}")
    batch_df.printSchema()
    print("  Sample (first 5 windows):")
    batch_df.orderBy("window_start").show(5, truncate=False)

# ── Streaming side ────────────────────────────────────────────────────────────
print()
if stream_df is None:
    print("⚠  Streaming output not found. Run Step 1 first.")
else:
    stream_count = stream_df.count()
    print("=== STREAMING — crashes_by_hour_window ===")
    print(f"  Rows (partial replay): {stream_count:,}")
    stream_df.printSchema()
    print("  Sample (first 5 windows):")
    stream_df.orderBy("window_start").show(5, truncate=False)

# ── Side-by-side column comparison ───────────────────────────────────────────
if batch_df is not None and stream_df is not None:
    print()
    print("=== Column Comparison ===")
    batch_cols   = set(batch_df.columns)
    stream_cols  = set(stream_df.columns)
    in_both      = sorted(batch_cols & stream_cols)
    only_batch   = sorted(batch_cols - stream_cols)
    only_stream  = sorted(stream_cols - batch_cols)

    print(f"  Shared columns  : {in_both}")
    print(f"  Batch-only      : {only_batch if only_batch else 'none'}")
    print(f"  Streaming-only  : {only_stream if only_stream else 'none'}")
    print()
    schemas_match = (only_batch == [] and only_stream == [])
    print("Schema parity:", "✓ MATCH" if schemas_match else "✗ MISMATCH — review columns above")

=== BATCH — crashes_by_hour_window ===
  Rows (full history) : 34,628
root
 |-- crashes: long (nullable = true)
 |-- injuries_total: long (nullable = true)
 |-- injuries_fatal: long (nullable = true)
 |-- window_start: timestamp (nullable = true)
 |-- window_end: timestamp (nullable = true)

  Sample (first 5 windows):
+-------+--------------+--------------+-------------------+-------------------+
|crashes|injuries_total|injuries_fatal|window_start       |window_end         |
+-------+--------------+--------------+-------------------+-------------------+
|12     |5             |0             |2022-01-01 00:00:00|2022-01-01 01:00:00|
|19     |3             |0             |2022-01-01 01:00:00|2022-01-01 02:00:00|
|21     |11            |0             |2022-01-01 02:00:00|2022-01-01 03:00:00|
|19     |11            |0             |2022-01-01 03:00:00|2022-01-01 04:00:00|
|13     |3             |0             |2022-01-01 04:00:00|2022-01-01 05:00:00|
+-------+--------------+--------------+

## Step 5 — Evidence Checklist for the Report

This checklist was used to capture all evidence needed for the batch vs. streaming behavioral comparison section of the report.

---

### A. Time-to-First-Output (Streaming Latency)

- Wall-clock timestamp when `dropper.py` drops the **first** file into `data/stream_in/` : 19:53:11
- Wall-clock timestamp when the **first `.parquet` file** appears in `outputs/streaming/`: 19:53:24
- **Time-to-first-output** = second timestamp − first timestamp = **13 seconds**!!
- Note:  In the batch model, the equivalent is zero (no output until the full job completes). This difference illustrates the fundamental latency trade-off.

---

### B. Checkpoint Path & Restart Behaviour

- Checkpoint directory: `data/checkpoints/streaming_job/`
- The checkpoint guarantees **exactly-once** processing: if `streaming_job.py` is stopped and restarted, it resumes from the last committed micro-batch offset and does not reprocess already-seen files.
- **To demonstrate restart:** we stopped `streaming_job.py` (Ctrl+C), drop a few more files via `dropper.py`, then restarted `streaming_job.py`. We confirm that only new files were processed (row counts increased by the expected delta, not from scratch).

---

### C. Watermark Behaviour

- Watermark: **1 hour** on `CRASH_DATE`.
- A window `[T, T+1h)` is finalised and emitted only after event-time `T + 1h + 1h = T + 2h` has been observed in the stream.
- Late-arriving events with `CRASH_DATE < (max observed event time − 1 hour)` are silently dropped.
- This is the standard Spark Structured Streaming trade-off between completeness and latency.

---

### D. Batch vs. Streaming Comparison Summary (for report table)

| Metric | Batch (`02_batch_analysis.ipynb`) | Streaming (`streaming_job.py`) |
|---|---|---|
| Processing model | Full scan → aggregate → write | Incremental micro-batch |
| First result available | After entire job completes | After first micro-batch (~trigger interval) |
| Window definition | `F.window("CRASH_DATE", "1 hour")` | `window(col("CRASH_DATE"), "1 hour")` |
| Watermark | N/A (full history, no late data) | 1 hour |
| Output mode | N/A (batch overwrite) | `append` |
| Total rows (full replay) | Complete (2022–2025) | Converges to batch total after all chunks dropped |

In [7]:
# Release resources — stop the SparkSession used for validation reads.
spark.stop()
print("SparkSession stopped. Streaming logic preview complete.")

SparkSession stopped. Streaming logic preview complete.
